<a href="https://colab.research.google.com/github/sabithakrishnan/fall_detection/blob/main/YOLO_Pose_%2B_Geometric_Feature_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ultralytics]


In [ ]:
import cv2
import os
import numpy as np
from ultralytics import YOLO

# Load the YOLO Pose model (Specifically built for skeleton tracking)
model = YOLO("yolo11n-pose.pt")

video_path = "/content/cam1.avi"
if not os.path.exists(video_path):
    raise FileNotFoundError(f"⚠️ Error: '{video_path}' not found! Please upload your video to Colab's left sidebar folder first.")

cap = cv2.VideoCapture(video_path)

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Setup Video Writer using XVID for AVI format
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('yolo_pose_fall_detection.avi', fourcc, fps, (frame_width, frame_height))

print("🎬 Processing video using Skeletal Pose Extraction...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Run Pose Inference
    results = model.predict(frame, conf=0.4, verbose=False)

    for result in results:
        # Check if any skeletons were detected in the frame
        if result.keypoints is not None and len(result.keypoints.data) > 0:

            # Iterate through each detected person
            for i, kpts in enumerate(result.keypoints.data):
                # kpts shape is (17, 3) -> 17 keypoints with [x, y, confidence]
                # COCO Keypoint Indexes: 5=Left Shoulder, 6=Right Shoulder, 11=Left Hip, 12=Right Hip

                # Extract coordinates
                l_shoulder = kpts[5].cpu().numpy()
                r_shoulder = kpts[6].cpu().numpy()
                l_hip = kpts[11].cpu().numpy()
                r_hip = kpts[12].cpu().numpy()

                # Ensure the model is confident about these vital tracking joints
                if l_shoulder[2] > 0.3 and r_shoulder[2] > 0.3 and l_hip[2] > 0.3 and r_hip[2] > 0.3:

                    # Calculate the center point of the shoulders
                    shoulder_center_x = (l_shoulder[0] + r_shoulder[0]) / 2
                    shoulder_center_y = (l_shoulder[1] + r_shoulder[1]) / 2

                    # Calculate the center point of the hips
                    hip_center_x = (l_hip[0] + r_hip[0]) / 2
                    hip_center_y = (l_hip[1] + r_hip[1]) / 2

                    # Calculate the angle of the spine relative to the horizontal floor plane
                    delta_x = shoulder_center_x - hip_center_x
                    delta_y = shoulder_center_y - hip_center_y

                    # np.arctan2 returns radians, convert it directly to degrees
                    spine_angle = abs(np.degrees(np.arctan2(delta_y, delta_x)))

                    # Get person's bounding box coordinates for drawing UI
                    if result.boxes is not None:
                        box = result.boxes[i].xyxy[0].cpu().numpy()
                        xmin, ymin, xmax, ymax = map(int, box)

                        # --- GEOMETRIC FALL THRESHOLD ---
                        # If the spine angle falls below 35 degrees, they are horizontal on the ground
                        if spine_angle < 35 or spine_angle > 145:
                            label = f"FALL DETECTED! Spine: {spine_angle:.1f}°"
                            color = (0, 0, 255)  # Red Alert
                        else:
                            label = "Person Standing"
                            color = (0, 255, 0)  # Green Normal

                        # Draw the bounding box and label
                        cv2.rectangle(frame, (xmin, ymin), (xmax, ymax), color, 3)
                        cv2.putText(frame, label, (xmin, ymin - 10),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

                        # Draw a blue tracking line representing the spine vector for visual confirmation
                        cv2.line(frame, (int(hip_center_x), int(hip_center_y)),
                                 (int(shoulder_center_x), int(shoulder_center_y)), (255, 0, 0), 4)

    out.write(frame)

cap.release()
out.release()
cv2.destroyAllWindows()
print("✅ Done! File saved as 'yolo_pose_fall_detection.avi'")


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
🎬 Processing video using Skeletal Pose Extraction...
✅ Done! File saved as 'yolo_pose_fall_detection.avi'


In [ ]:
import os
from IPython.display import HTML
from base64 import b64encode

#Convert the new pose output video to a web-optimized MP4 file using FFmpeg
print("Converting pose estimation video for browser playback...")
os.system("ffmpeg -y -i yolo_pose_fall_detection.avi -vcodec libx264 -f mp4 pose_preview.mp4")

#Read the converted video file and encode it to a Base64 data string
video_file = open('pose_preview.mp4', 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(video_file).decode()

#Create and embed the HTML5 video player canvas inside the Colab cell output
print("Done! You can watch the pose tracking video below:")
HTML(f"""
<video width="720" controls>
      <source src="{data_url}" type="video/mp4">
</video>
""")


Converting pose estimation video for browser playback...
Done! You can watch the pose tracking video below:
